# Lab 2.1 - Find Similar Documents with Embeddings (Solution)
**Day 2 - Session 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/lab2_1_similarity/solution/lab2_1_similarity.ipynb)

**Goal:** Embed a small set of documents with a sentence-transformers model,
then rank them by cosine similarity to a query. Compare semantic search with
plain keyword matching.

This solution has the full working code. Everything runs on a Colab CPU
runtime in under a minute; no API key and no GPU needed.

In [ ]:
# Scoped install for this lab (safe to re-run; preinstalled on Colab).
%pip install -q sentence-transformers scikit-learn

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

np.random.seed(42)

## 1. A small document collection
Eight short help-desk style documents. Small enough to read, varied enough to
show that meaning beats keyword overlap.

In [ ]:
DOCUMENTS = [
    "How to reset your account password from the login screen.",
    "Steps to update your billing information and payment method.",
    "Troubleshooting slow application startup on older laptops.",
    "How to invite a teammate and manage their permissions.",
    "Recovering access when you are locked out of your account.",
    "Exporting your monthly invoices as a PDF file.",
    "Fixing sync errors between the mobile app and the web app.",
    "Changing your notification settings and email preferences.",
]
for i, doc in enumerate(DOCUMENTS):
    print(f"{i}: {doc}")

## 2. Load an embedding model
`all-MiniLM-L6-v2` is small (about 80 MB) and fast on CPU. It maps each text
to a 384-dimension dense vector.

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = model.encode(DOCUMENTS, normalize_embeddings=True)
print("Embeddings shape:", doc_embeddings.shape)

## 3. Rank documents by cosine similarity
We normalized the vectors, so cosine similarity is just a dot product. Higher
score means closer in meaning.

In [ ]:
def search(query, top_k=3):
    query_vec = model.encode([query], normalize_embeddings=True)[0]
    scores = doc_embeddings @ query_vec
    ranked = np.argsort(scores)[::-1]
    print(f"\nQuery: {query!r}")
    for rank, idx in enumerate(ranked[:top_k], start=1):
        print(f"  {rank}. score={scores[idx]:.3f}  {DOCUMENTS[idx]}")
    return ranked, scores


_ = search("I forgot my password and cannot log in")

## 4. Semantic search vs keyword matching
The query below shares no keywords with the best document. Keyword matching
misses it; embeddings still find it because the meaning is close.

In [ ]:
def keyword_search(query, top_k=3):
    query_words = set(query.lower().split())
    overlap = [len(query_words & set(doc.lower().split())) for doc in DOCUMENTS]
    ranked = np.argsort(overlap)[::-1]
    print(f"\nKeyword query: {query!r}")
    for rank, idx in enumerate(ranked[:top_k], start=1):
        print(f"  {rank}. overlap={overlap[idx]}  {DOCUMENTS[idx]}")
    return ranked


QUERY = "locked out and need to regain entry"
print("=== Semantic search ===")
search(QUERY)
print("\n=== Keyword search ===")
keyword_search(QUERY)

Semantic search ranks "Recovering access when you are locked out of your
account" near the top. Keyword search struggles because the wording differs.

## Stretch goals
1. Cluster the documents with KMeans and print each cluster.
2. Try a second embedding model and compare the top result for a query.

### Stretch 1: cluster the documents

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(doc_embeddings)
for cluster_id in sorted(set(labels)):
    print(f"\nCluster {cluster_id}:")
    for idx, label in enumerate(labels):
        if label == cluster_id:
            print(f"  - {DOCUMENTS[idx]}")

### Stretch 2: compare a second model

In [ ]:
model2 = SentenceTransformer("all-MiniLM-L12-v2")
doc_embeddings2 = model2.encode(DOCUMENTS, normalize_embeddings=True)
query_vec2 = model2.encode(["I forgot my password and cannot log in"],
                           normalize_embeddings=True)[0]
best2 = int(np.argmax(doc_embeddings2 @ query_vec2))
print("MiniLM-L12 best match:", DOCUMENTS[best2])

## Expected output
- The password query returns the password-reset and account-recovery docs first.
- Semantic search beats keyword search on the "locked out" query.
- KMeans groups the documents into account, billing, and app-issue themes.
- Both embedding models return a sensible top match for the password query.